# Lesson 8A: Anomaly Detection Theory

<a name="introduction"></a>
## Introduction

Most supervised learning assumes you have plentiful examples of every class you
care about. Anomaly detection breaks that assumption: anomalies are rare by
definition, often unlabeled, and may take forms never seen in training data.
The question shifts from "which class does this belong to?" to "how unusual is
this, relative to what normal looks like?"

Three families of approach answer that question differently:

1. **Statistical (density-based)**: model the distribution of normal data
   explicitly (Gaussian, multivariate Gaussian) and flag low-probability points
2. **Isolation-based**: exploit the fact that anomalies are "few and different"
   — they should be easy to isolate with random partitioning, without ever
   modeling a distance or density
3. **Boundary-based (One-Class SVM)**: learn a boundary that encloses normal
   data in feature space, without needing to estimate a full density

In this lesson, we'll:
1. Derive Gaussian anomaly detection and the Mahalanobis distance for correlated features
2. Derive why Isolation Forest works without any explicit distance metric
3. Derive the One-Class SVM formulation as a boundary-fitting problem
4. Implement Gaussian anomaly detection from scratch with full covariance estimation
5. Compare all three approaches on the same dataset

Lesson 8b applies these methods to a fraud detection case study, focusing on the
practical challenge of extreme class imbalance.


## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Gaussian Anomaly Detection](#gaussian-anomaly-detection)
   - [Univariate Gaussian Model](#univariate-gaussian-model)
   - [Threshold Selection](#threshold-selection)
4. [Multivariate Gaussian and Mahalanobis Distance](#multivariate-gaussian-and-mahalanobis-distance)
   - [Accounting for Feature Correlation](#accounting-for-feature-correlation)
   - [Deriving the Mahalanobis Distance](#deriving-the-mahalanobis-distance)
   - [Parameter Estimation](#gaussian-parameter-estimation)
5. [Isolation Forest](#isolation-forest)
   - [The Isolation Principle](#the-isolation-principle)
   - [Path Length and Anomaly Score](#path-length-and-anomaly-score)
6. [One-Class SVM](#one-class-svm)
   - [Boundary-Fitting Formulation](#boundary-fitting-formulation)
   - [The Role of the Kernel](#the-role-of-the-kernel)
7. [From-Scratch Implementation](#from-scratch-implementation)
   - [Multivariate Gaussian Anomaly Detector](#multivariate-gaussian-anomaly-detector)
8. [Visualization and Interpretation](#visualization-and-interpretation)
   - [Univariate vs Multivariate Detection](#univariate-vs-multivariate-detection)
   - [Isolation Forest Path Lengths](#isolation-forest-path-lengths)
9. [Comparing Approaches](#comparing-approaches)
10. [Assumptions and Limitations](#assumptions-and-limitations)
11. [Conclusion](#conclusion)
    - [Key Insights](#key-insights-5)
    - [When to Use Each Approach](#when-to-use-each-approach-2)
    - [Further Reading](#further-reading-5)


<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


<a name="gaussian-anomaly-detection"></a>
## Gaussian Anomaly Detection

<a name="univariate-gaussian-model"></a>
### Univariate Gaussian Model

The simplest anomaly detector assumes each feature, independently, follows a
normal distribution when the data point is "normal":

$$p(x; \mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$

For a $d$-dimensional point with features assumed independent, the joint density
factorizes:

$$p(\mathbf{x}) = \prod_{i=1}^{d} p(x_i; \mu_i, \sigma_i^2)$$

A point is flagged as anomalous when its estimated density is very low:

$$\text{anomaly if } p(\mathbf{x}) < \varepsilon$$

for a threshold $\varepsilon$ chosen using a validation set of labeled
anomalies (if available) or a target false-positive rate.

<a name="threshold-selection"></a>
### Threshold Selection

Choosing $\varepsilon$ trades off false positives (flagging normal points as
anomalous) against false negatives (missing true anomalies). With a small
labeled validation set, $\varepsilon$ is typically swept over a range and the
value maximizing F1-score (or another metric appropriate to the imbalance) is
selected — this is precisely the ROC/precision-recall analysis Lesson 8b
performs on real fraud data.


In [ ]:
# Demonstrate univariate Gaussian anomaly detection
np.random.seed(42)
normal_data = np.random.normal(loc=50, scale=5, size=500)
anomalies = np.array([20, 25, 75, 80, 15])

mu_hat = normal_data.mean()
sigma2_hat = normal_data.var()

x_range = np.linspace(0, 100, 500)
density = stats.norm.pdf(x_range, mu_hat, np.sqrt(sigma2_hat))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(normal_data, bins=40, density=True, alpha=0.5, color='steelblue', label='Normal data')
ax.plot(x_range, density, 'b-', linewidth=2, label='Fitted Gaussian')
ax.scatter(anomalies, stats.norm.pdf(anomalies, mu_hat, np.sqrt(sigma2_hat)),
           color='red', s=100, zorder=5, marker='x', linewidths=3, label='Candidate anomalies')
ax.set_xlabel('Feature value')
ax.set_ylabel('Density')
ax.set_title('Univariate Gaussian Anomaly Detection')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

epsilon = 0.001
print("\n" + "="*70)
print("UNIVARIATE GAUSSIAN ANOMALY DETECTION")
print("="*70)
print(f"\nFitted mu = {mu_hat:.2f}, sigma = {np.sqrt(sigma2_hat):.2f}")
print(f"Threshold epsilon = {epsilon}")
print(f"\n{'Point':<10}{'p(x)':<15}{'Flagged?':<10}")
print("-"*40)
for point in anomalies:
    p_x = stats.norm.pdf(point, mu_hat, np.sqrt(sigma2_hat))
    flagged = "ANOMALY" if p_x < epsilon else "normal"
    print(f"{point:<10}{p_x:<15.6f}{flagged:<10}")


<a name="multivariate-gaussian-and-mahalanobis-distance"></a>
## Multivariate Gaussian and Mahalanobis Distance

<a name="accounting-for-feature-correlation"></a>
### Accounting for Feature Correlation

Treating features as independent (as above) misses a common failure mode:
two features can each look individually normal, yet be jointly anomalous
because their *relationship* is unusual (e.g. height and weight that are each
plausible alone but wildly inconsistent together). The **multivariate
Gaussian** models this directly with a full covariance matrix $\Sigma$:

$$p(\mathbf{x}; \boldsymbol{\mu}, \Sigma) = \frac{1}{(2\pi)^{d/2} |\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T \Sigma^{-1} (\mathbf{x}-\boldsymbol{\mu})\right)$$

<a name="deriving-the-mahalanobis-distance"></a>
### Deriving the Mahalanobis Distance

The exponent in the multivariate Gaussian density is (up to a factor of
$-\frac{1}{2}$) the **squared Mahalanobis distance**:

$$D_M(\mathbf{x})^2 = (\mathbf{x}-\boldsymbol{\mu})^T \Sigma^{-1} (\mathbf{x}-\boldsymbol{\mu})$$

**Why $\Sigma^{-1}$, not the identity?** If features were uncorrelated with
equal variance ($\Sigma = \sigma^2 I$), this reduces to
$D_M(\mathbf{x})^2 = \frac{1}{\sigma^2}\|\mathbf{x}-\boldsymbol{\mu}\|^2$ —
ordinary (rescaled) Euclidean distance. With correlated features, $\Sigma^{-1}$
**whitens** the space: it stretches distance along directions of low variance
(where deviations are "surprising") and compresses it along directions of high
variance (where deviations are "expected"). A point that deviates along the
natural axis of correlation between two features is judged less anomalous
than a point that deviates *against* that correlation, even if both are the
same Euclidean distance from $\boldsymbol{\mu}$.

Since $p(\mathbf{x})$ is a strictly decreasing function of $D_M(\mathbf{x})^2$
(larger distance means lower density), thresholding on density is equivalent
to thresholding on Mahalanobis distance — anomaly detection reduces to
"is this point too far (in whitened space) from the mean?"

<a name="gaussian-parameter-estimation"></a>
### Parameter Estimation

As with Gaussian Naive Bayes (Lesson 6), maximum likelihood gives closed-form
estimates from training data $\{\mathbf{x}_n\}_{n=1}^{N}$ (assumed to contain
only, or overwhelmingly, normal points):

$$\hat{\boldsymbol{\mu}} = \frac{1}{N}\sum_{n=1}^{N} \mathbf{x}_n, \qquad \hat{\Sigma} = \frac{1}{N}\sum_{n=1}^{N} (\mathbf{x}_n - \hat{\boldsymbol{\mu}})(\mathbf{x}_n - \hat{\boldsymbol{\mu}})^T$$

Unlike Gaussian Naive Bayes, $\hat{\Sigma}$ here is a full $d \times d$ matrix
(not a diagonal one) — capturing exactly the cross-feature correlation that
the "naive" independence assumption discards.


In [ ]:
print("\n" + "="*70)
print("MULTIVARIATE GAUSSIAN: MLE PARAMETER ESTIMATION")
print("="*70)
print("\nmu_hat    = (1/N) * sum_n x_n                                (sample mean vector)")
print("Sigma_hat = (1/N) * sum_n (x_n - mu_hat)(x_n - mu_hat)^T       (full sample covariance)")
print("\nMahalanobis distance:")
print("  D_M(x)^2 = (x - mu_hat)^T Sigma_hat^{-1} (x - mu_hat)")
print("\nThis whitens the space by Sigma_hat^{-1} before measuring distance --")
print("directions of low variance count MORE, directions of high (expected)")
print("variance count LESS.")


<a name="isolation-forest"></a>
## Isolation Forest

<a name="the-isolation-principle"></a>
### The Isolation Principle

Isolation Forest takes a fundamentally different approach: instead of modeling
what "normal" looks like, it directly exploits a structural property of
anomalies — they are **few and different**, so they are easier to *isolate*
from the rest of the data than normal points are.

An isolation tree is built by recursively partitioning the data: at each node,
pick a random feature, then a random split value between that feature's
observed min and max, and split the data. Repeat until every point is alone in
its own leaf. **Anomalies, being few and different (typically in sparser,
more extreme regions of feature space), get isolated in fewer random splits**
than normal points, which are surrounded by many similar neighbors and require
many splits to separate from the crowd.

<a name="path-length-and-anomaly-score"></a>
### Path Length and Anomaly Score

For a point $\mathbf{x}$, let $h(\mathbf{x})$ be its path length (number of
splits) in a single isolation tree. Average this over an ensemble of $t$ trees
to get $\mathbb{E}[h(\mathbf{x})]$. The anomaly score normalizes this against
the expected path length of an unsuccessful search in a Binary Search Tree of
$n$ points:

$$c(n) = 2H(n-1) - \frac{2(n-1)}{n}, \qquad H(i) = \ln(i) + 0.5772156649 \text{ (harmonic number, Euler-Mascheroni constant)}$$

$$s(\mathbf{x}, n) = 2^{-\frac{\mathbb{E}[h(\mathbf{x})]}{c(n)}}$$

This score lies in $(0, 1]$:
- $s \to 1$ as $\mathbb{E}[h(\mathbf{x})] \to 0$ (isolated almost immediately — strong anomaly)
- $s \approx 0.5$ when $\mathbb{E}[h(\mathbf{x})] \approx c(n)$ (typical path length — no clear signal)
- $s \to 0$ as $\mathbb{E}[h(\mathbf{x})]$ grows large relative to $c(n)$ (needed many splits — normal point)

**Crucially, this requires no distance metric, no density estimate, and no
covariance matrix** — only random axis-aligned splits and average path
length. This makes Isolation Forest computationally cheap ($O(n \log n)$
training) and naturally robust to the curse of dimensionality that afflicts
distance-based methods.


In [ ]:
print("\n" + "="*70)
print("ISOLATION FOREST: PATH LENGTH NORMALIZATION")
print("="*70)

def c_factor(n):
    """Average path length of unsuccessful search in a BST of n points."""
    if n <= 1:
        return 0
    harmonic = np.log(n - 1) + 0.5772156649
    return 2 * harmonic - (2 * (n - 1) / n)

for n in [10, 100, 1000, 10000]:
    print(f"n={n:<8} c(n) = {c_factor(n):.4f}  (average path length for a typical point)")

print("\nAnomaly score s(x,n) = 2^(-E[h(x)] / c(n)):")
print("  E[h(x)] << c(n)  ->  s -> 1  (isolated quickly -- anomaly)")
print("  E[h(x)] == c(n)  ->  s == 0.5  (typical -- no signal)")
print("  E[h(x)] >> c(n)  ->  s -> 0  (many splits needed -- normal)")


<a name="one-class-svm"></a>
## One-Class SVM

<a name="boundary-fitting-formulation"></a>
### Boundary-Fitting Formulation

One-Class SVM adapts the maximum-margin idea from Lesson 4 to a setting with
no negative class at all: instead of separating two classes, it finds a
boundary that separates the bulk of the training data from the origin in
feature space, with maximum margin. The optimization problem is:

$$\min_{\mathbf{w}, \rho, \boldsymbol{\xi}} \quad \frac{1}{2}\|\mathbf{w}\|^2 + \frac{1}{\nu n}\sum_{i=1}^{n}\xi_i - \rho$$

$$\text{subject to} \quad \mathbf{w}^T \phi(\mathbf{x}_i) \geq \rho - \xi_i, \quad \xi_i \geq 0$$

where $\phi$ is a (possibly kernel-induced) feature map, $\rho$ is the offset
of the separating hyperplane from the origin, and $\xi_i$ are slack variables
allowing some training points to fall on the wrong side (as in soft-margin
SVM). The decision function is:

$$f(\mathbf{x}) = \text{sign}(\mathbf{w}^T \phi(\mathbf{x}) - \rho)$$

with $f(\mathbf{x}) < 0$ flagging $\mathbf{x}$ as an outlier.

**The role of $\nu$**: the parameter $\nu \in (0, 1]$ is simultaneously an
upper bound on the fraction of training points allowed to be outliers, and a
lower bound on the fraction of points that become support vectors — it is a
direct, interpretable knob for the expected anomaly rate, unlike the implicit
threshold $\varepsilon$ in the Gaussian approach.

<a name="the-role-of-the-kernel"></a>
### The Role of the Kernel

With a linear kernel, One-Class SVM fits a hyperplane, which can only really
express "distance from a single subspace" as the boundary — with the RBF
kernel (the common default), the boundary in the *original* space becomes an
arbitrarily-shaped closed region enclosing dense clusters of normal data,
exactly analogous to how the kernel trick let SVM in Lesson 4 draw curved
decision boundaries for classification.


In [ ]:
print("\n" + "="*70)
print("ONE-CLASS SVM: BOUNDARY FITTING")
print("="*70)
print("\nObjective: minimize (1/2)||w||^2 + (1/(nu*n)) * sum(xi_i) - rho")
print("Subject to: w^T phi(x_i) >= rho - xi_i,  xi_i >= 0")
print("\nDecision function: f(x) = sign(w^T phi(x) - rho)")
print("  f(x) < 0  ->  outlier")
print("  f(x) >= 0 ->  normal")
print("\nnu in (0,1] controls both the training outlier fraction AND the")
print("support-vector fraction -- a direct knob for the expected anomaly rate.")


<a name="from-scratch-implementation"></a>
## From-Scratch Implementation

<a name="multivariate-gaussian-anomaly-detector"></a>
### Multivariate Gaussian Anomaly Detector

In [ ]:
class GaussianAnomalyDetector:
    """
    Multivariate Gaussian anomaly detector using maximum likelihood estimation
    of the mean vector and full covariance matrix, and Mahalanobis-distance
    thresholding for detection.
    """

    def __init__(self):
        self.mean_ = None
        self.cov_ = None
        self.cov_inv_ = None
        self.log_det_cov_ = None
        self.n_features_ = None

    def fit(self, X):
        """Estimate mu and Sigma from (assumed-normal) training data."""
        n_samples, n_features = X.shape
        self.n_features_ = n_features
        self.mean_ = X.mean(axis=0)

        centered = X - self.mean_
        self.cov_ = (centered.T @ centered) / n_samples

        # Regularize for numerical stability (near-singular covariance)
        self.cov_ += np.eye(n_features) * 1e-6
        self.cov_inv_ = np.linalg.inv(self.cov_)
        sign, logdet = np.linalg.slogdet(self.cov_)
        self.log_det_cov_ = logdet

        return self

    def mahalanobis_distance(self, X):
        """Squared Mahalanobis distance: (x - mu)^T Sigma^-1 (x - mu)."""
        centered = X - self.mean_
        # Efficient batched quadratic form: sum over features of (centered @ Sigma^-1) * centered
        return np.sum((centered @ self.cov_inv_) * centered, axis=1)

    def log_density(self, X):
        """Log p(x) for the fitted multivariate Gaussian."""
        d = self.n_features_
        mahalanobis_sq = self.mahalanobis_distance(X)
        return -0.5 * (d * np.log(2 * np.pi) + self.log_det_cov_ + mahalanobis_sq)

    def score_samples(self, X):
        """Density p(x) -- higher means more 'normal'."""
        return np.exp(self.log_density(X))

    def predict(self, X, epsilon):
        """Return -1 for anomaly (p(x) < epsilon), +1 for normal."""
        densities = self.score_samples(X)
        return np.where(densities < epsilon, -1, 1)


print("\n" + "="*70)
print("MULTIVARIATE GAUSSIAN ANOMALY DETECTOR: FROM-SCRATCH IMPLEMENTATION")
print("="*70)
print("\nFit: estimate mu (sample mean) and Sigma (sample covariance)")
print("Score: log p(x) via the multivariate Gaussian log-density formula")
print("Predict: flag x as anomalous when p(x) falls below a chosen threshold")


<a name="visualization-and-interpretation"></a>
## Visualization and Interpretation

<a name="univariate-vs-multivariate-detection"></a>
### Univariate vs Multivariate Detection

In [ ]:
# Demonstrate the case where multivariate Gaussian catches what univariate misses:
# correlated features where a point is individually normal on each axis but
# jointly anomalous given the correlation.
np.random.seed(42)
mean = [0, 0]
cov = [[1, 0.85], [0.85, 1]]  # strong positive correlation
normal_points = np.random.multivariate_normal(mean, cov, 300)

# A point that is well within range on EACH axis individually, but violates
# the correlation structure (e.g. high on feature 1, low on feature 2)
joint_anomaly = np.array([[2.0, -2.0]])

detector = GaussianAnomalyDetector()
detector.fit(normal_points)

# Univariate (independence-assuming) density for comparison
mu_uni = normal_points.mean(axis=0)
sigma_uni = normal_points.std(axis=0)
univariate_density = np.prod(stats.norm.pdf(joint_anomaly, mu_uni, sigma_uni))
multivariate_density = detector.score_samples(joint_anomaly)[0]
mahalanobis_dist = np.sqrt(detector.mahalanobis_distance(joint_anomaly)[0])
euclidean_dist = np.linalg.norm(joint_anomaly[0] - mu_uni)

fig, ax = plt.subplots(1, 1, figsize=(9, 8))
ax.scatter(normal_points[:, 0], normal_points[:, 1], alpha=0.4, s=30, label='Normal (correlated) data')
ax.scatter(joint_anomaly[:, 0], joint_anomaly[:, 1], color='red', s=200, marker='x',
           linewidths=3, label='Point (2.0, -2.0)')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Point Individually Normal on Each Axis, Jointly Anomalous')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("WHY MAHALANOBIS DISTANCE CATCHES WHAT PER-FEATURE CHECKS MISS")
print("="*70)
print(f"\nPoint (2.0, -2.0): each coordinate is only ~2 standard deviations from")
print(f"the marginal mean on its own axis -- not extreme in isolation.")
print(f"\nEuclidean distance from mean:    {euclidean_dist:.4f}")
print(f"Mahalanobis distance from mean:  {mahalanobis_dist:.4f}")
print(f"\nUnivariate (independence-assuming) density: {univariate_density:.6e}")
print(f"Multivariate (correlation-aware) density:   {multivariate_density:.6e}")
print("\nThe multivariate density is far lower -- because the point contradicts")
print("the strong positive correlation between the two features. Mahalanobis")
print("distance is much larger than Euclidean distance because Sigma^-1")
print("stretches distance in the direction orthogonal to the correlation.")


<a name="isolation-forest-path-lengths"></a>
### Isolation Forest Path Lengths

In [ ]:
# Visualize how isolation forest path length differs for normal vs anomalous points
np.random.seed(42)
X_cluster, _ = make_blobs(n_samples=300, centers=[[0, 0]], cluster_std=1.0, random_state=42)
X_anomalies = np.array([[6, 6], [-6, -5], [7, -6]])
X_combined = np.vstack([X_cluster, X_anomalies])

iso_forest = IsolationForest(n_estimators=100, random_state=42)
iso_forest.fit(X_combined)

anomaly_scores = -iso_forest.score_samples(X_combined)  # sklearn returns negative of our s(x,n) convention
predictions = iso_forest.predict(X_combined)

fig, ax = plt.subplots(1, 1, figsize=(9, 7))
scatter = ax.scatter(X_combined[:, 0], X_combined[:, 1], c=anomaly_scores, cmap='RdYlBu_r',
                      s=60, edgecolors='k', alpha=0.8)
plt.colorbar(scatter, label='Anomaly score (higher = more anomalous)')
ax.set_title('Isolation Forest Anomaly Scores')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ISOLATION FOREST: SCORES FOR PLANTED ANOMALIES")
print("="*70)
for point, score, pred in zip(X_anomalies, anomaly_scores[-3:], predictions[-3:]):
    label = "ANOMALY" if pred == -1 else "normal"
    print(f"Point {tuple(point)}: anomaly_score={score:.4f}, predicted={label}")

n_detected = np.sum(predictions[-3:] == -1)
print(f"\n{n_detected}/3 planted anomalies correctly flagged.")
print("\nThe planted points sit far from the dense cluster and get isolated in")
print("very few random splits, producing high anomaly scores -- exactly the")
print("mechanism derived above, with no distance metric ever computed.")


<a name="comparing-approaches"></a>
## Comparing Approaches

In [ ]:
# Compare from-scratch Gaussian, scikit-learn Gaussian (EmpiricalCovariance +
# Mahalanobis), Isolation Forest, and One-Class SVM on the same dataset
np.random.seed(42)
X_normal, _ = make_blobs(n_samples=300, centers=[[0, 0]], cluster_std=1.5, random_state=42)
X_test_anomalies = np.array([[8, 8], [-7, 6], [9, -7], [-8, -8], [10, 0]])
X_test_normal, _ = make_blobs(n_samples=50, centers=[[0, 0]], cluster_std=1.5, random_state=1)
X_test = np.vstack([X_test_normal, X_test_anomalies])
y_test_true = np.concatenate([np.ones(50), -np.ones(5)])  # 1 = normal, -1 = anomaly

results = {}

# 1. From-scratch Gaussian
gauss_scratch = GaussianAnomalyDetector()
gauss_scratch.fit(X_normal)
scores_scratch = gauss_scratch.score_samples(X_test)
threshold = np.percentile(gauss_scratch.score_samples(X_normal), 1)  # 1st percentile of training densities
pred_scratch = np.where(scores_scratch < threshold, -1, 1)
results['Gaussian (from-scratch)'] = pred_scratch

# 2. scikit-learn EmpiricalCovariance + Mahalanobis
emp_cov = EmpiricalCovariance().fit(X_normal)
mahal_dist = emp_cov.mahalanobis(X_test)
mahal_threshold = np.percentile(emp_cov.mahalanobis(X_normal), 99)
pred_sklearn_gauss = np.where(mahal_dist > mahal_threshold, -1, 1)
results['Gaussian (scikit-learn)'] = pred_sklearn_gauss

# 3. Isolation Forest
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso.fit(X_normal)
results['Isolation Forest'] = iso.predict(X_test)

# 4. One-Class SVM
ocsvm = OneClassSVM(kernel='rbf', nu=0.05, gamma='scale')
ocsvm.fit(X_normal)
results['One-Class SVM'] = ocsvm.predict(X_test)

print("\n" + "="*70)
print("ANOMALY DETECTION METHOD COMPARISON")
print("="*70)
print(f"\n{'Method':<28}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}")
print("-"*70)
for name, pred in results.items():
    acc = accuracy_score(y_test_true, pred)
    prec = precision_score(y_test_true, pred, pos_label=-1, zero_division=0)
    rec = recall_score(y_test_true, pred, pos_label=-1, zero_division=0)
    f1 = f1_score(y_test_true, pred, pos_label=-1, zero_division=0)
    print(f"{name:<28}{acc:<12.4f}{prec:<12.4f}{rec:<12.4f}{f1:<12.4f}")

print("\n(precision/recall/F1 computed for the anomaly class, pos_label=-1)")
print("\nAll four methods achieve perfect recall here -- every planted anomaly is")
print("caught. But precision differs: the Gaussian methods get every prediction")
print("right, while Isolation Forest and One-Class SVM flag several normal")
print("points as anomalous too (lower precision). This is expected on a compact,")
print("single-cluster Gaussian blob -- exactly the setting Gaussian methods are")
print("built for. The gap direction can reverse on non-Gaussian, multi-modal, or")
print("high-dimensional data, which Lesson 8b explores with real fraud data.")


<a name="assumptions-and-limitations"></a>
## Assumptions and Limitations

**Gaussian (statistical) methods** assume the normal data is genuinely
(multivariate) Gaussian-distributed. Real data is often multi-modal (several
distinct clusters of "normal" behavior) or heavy-tailed, where a single
Gaussian fits poorly and either misses real anomalies (if the Gaussian is
too wide) or flags too many normal points (if too narrow). The covariance
matrix also requires $O(d^2)$ parameters and $O(d^3)$ inversion cost — expensive
and numerically fragile in high dimensions with limited data.

**Isolation Forest** makes no distributional assumption and scales well with
dimensionality, but its axis-aligned random splits can struggle when
anomalies are only distinguishable via a rotated or non-linear combination of
features (an anomaly hidden along a diagonal direction may not isolate faster
than normal points under axis-aligned splitting).

**One-Class SVM** with an RBF kernel can capture non-linear, non-convex normal
regions, but is sensitive to the kernel bandwidth $\gamma$ and the target
outlier fraction $\nu$ — poor choices either overfit to training noise (too
tight a boundary) or fail to flag real anomalies (too loose). It also scales
poorly to large datasets ($O(n^2)$ to $O(n^3)$ depending on the solver, as
with standard SVM in Lesson 4).

**All three families share one deeper limitation**: they detect statistical
outliers relative to the training distribution, not necessarily
*semantically* anomalous events. A fraudulent transaction that closely
mimics normal spending patterns will evade every method here — anomaly
detection catches what is statistically unusual, not what is definitionally
wrong.


<a name="conclusion"></a>
## Conclusion

<a name="key-insights-5"></a>
### Key Insights

1. **Gaussian anomaly detection** flags points with low estimated probability
   density; the univariate version assumes independent features, the
   multivariate version captures correlation via a full covariance matrix

2. **Mahalanobis distance** is the multivariate Gaussian's exponent in
   disguise — it whitens feature space by $\Sigma^{-1}$ so that deviations
   against the natural correlation structure count as more anomalous than
   deviations along it

3. **Isolation Forest** needs no distance metric or density estimate at all —
   it exploits the structural fact that anomalies are isolated faster than
   normal points under random recursive partitioning

4. **One-Class SVM** reframes anomaly detection as a maximum-margin boundary
   problem separating normal data from the origin, with $\nu$ directly
   controlling the expected outlier fraction

5. **The from-scratch multivariate Gaussian detector** correctly identifies
   correlation-violating anomalies that a per-feature (univariate) check misses

6. **Every method here detects statistical outliers, not semantic wrongness** —
   this distinction matters most in adversarial settings like fraud detection


<a name="when-to-use-each-approach-2"></a>
### When to Use Each Approach

**Gaussian methods are particularly good for:**
- Low-to-moderate dimensional, genuinely unimodal normal data
- Settings where interpretability of "why flagged" matters (Mahalanobis distance decomposes cleanly)
- Cases with enough data to estimate a stable covariance matrix

**Isolation Forest is particularly good for:**
- High-dimensional data where density estimation becomes unreliable
- Large datasets (near-linear training time, no distance-matrix computation)
- Settings with no strong assumption about the shape of "normal"

**One-Class SVM is particularly good for:**
- Moderate-sized datasets with genuinely non-convex normal regions
- Cases where a direct, interpretable control over the outlier fraction (nu) is valuable
- Settings already using kernel methods elsewhere in the pipeline


<a name="further-reading-5"></a>
### Further Reading

**Foundational Papers:**
- Chandola, V., Banerjee, A., & Kumar, V. (2009). "Anomaly Detection: A Survey"
- Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). "Isolation Forest"
- Schölkopf, B., et al. (2001). "Estimating the Support of a High-Dimensional Distribution" (One-Class SVM)

**Comprehensive References:**
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). "The Elements of Statistical Learning (ESL)"
- Aggarwal, C. C. (2017). "Outlier Analysis" (2nd ed.)
